# Modelo de Clasificación y Predicción de Anemia Infantil mediante LDA
**Proyecto Final de Curso — Ingeniería de Sistemas (UNSCH)**  
* **Dataset:** ENDES 2025 (Módulo RECH6: Antropometría y Salud en Menores de 5 años)  
* **Algoritmo:** Análisis Discriminante Lineal (LDA)

---

## 1. Configuración del Entorno e Importación de Librerías
Se cargan las librerías esenciales para el procesamiento numérico, manipulación de estructuras tabulares, gráficos y tablas interactivas.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import HTML, display
display(HTML("<script>document.documentElement.classList.add('dark');</script>"))
from itables import show

## 2. Carga del Dataset y Selección de Variables
Cargamos el archivo base `RECH6_2025.csv`. Definimos explícitamente:
1. **Variable Objetivo ($Y$):** `HC57` (Nivel de anemia según la normativa vigente).
2. **Variables Predictoras ($X$):** Factores antropométricos, biológicos y sociodemográficos seleccionados teóricamente para **evitar el sesgo de fuga de datos (*data leakage*)**, excluyendo intencionalmente los niveles directos de hemoglobina.

In [55]:
# Cargar el dataset del módulo RECH6 de la ENDES 2025
df = pd.read_csv("data/RECH6_2025.csv", low_memory=False)

print(f"Dimensiones originales del dataset cargado: {df.shape}")

# Definir la variable objetivo y las predictoras teóricamente justificadas
variable_objetivo = 'HC57'
variables_predictoras = [
    'HC1',   # Edad en meses (Vulnerabilidad por etapa de crecimiento)
    'HC2',   # Peso en kg (Estado nutricional general)
    'HC3',   # Talla en cm (Crecimiento lineal)
    'HC70',  # Puntaje Z Talla/Edad (Desnutrición crónica)
    'HC71',  # Puntaje Z Peso/Edad (Desnutrición global)
    'HC72',  # Puntaje Z Peso/Talla (Desnutrición aguda)
    'HC73',  # Puntaje Z IMC (Composición corporal)
    'HC27',  # Sexo del niño (Control demográfico)
    'HC61',  # Nivel educativo de la madre (Entorno socioeducativo)
    'HC62',  # Años de educación de la madre (Precisión numérica educativa)
    'HC64'   # Orden de nacimiento (Dinámica familiar)
]

# Crear el subconjunto de trabajo
columnas_estudio = variables_predictoras + [variable_objetivo]
df_subset = df[columnas_estudio].copy()

# Mostrar una vista interactiva de los primeros registros
show(df_subset)

Dimensiones originales del dataset cargado: (19298, 43)


## 3. Renombrado y Limpieza de Datos
Para ganar legibilidad del modelo:
1. **Renombramos las variables técnicas** (`HC1`, `HC2`, etc.) a nombres descriptivos en español.
2. **Tratamiento de códigos especiales:** La ENDES utiliza códigos numéricos específicos para valores perdidos o rechazos (ej. `96`, `98`, `99`, `9996`, `9998`). Estos se convierten en nulos (`NaN`).
3. **Gestión de valores nulos:** Se eliminan los registros que no cuenten con la etiqueta de la variable objetivo (`nivel_anemia`) o con nulos en las variables predictoras.

In [56]:
# 1. Renombrar las columnas técnicas a nombres legibles
renombres = {
    'HC1': 'edad_meses',
    'HC2': 'peso_kg',
    'HC3': 'talla_cm',
    'HC70': 'z_talla_edad',
    'HC71': 'z_peso_edad',
    'HC72': 'z_peso_talla',
    'HC73': 'z_imc',
    'HC27': 'sexo_nino',
    'HC61': 'educacion_madre_nivel',
    'HC62': 'educacion_madre_anios',
    'HC64': 'orden_nacimiento',
    'HC57': 'nivel_anemia'
}

df_subset = df_subset.rename(columns=renombres)

show(df_subset)

In [57]:
# Crear un resumen de los tipos de datos de cada columna
df_tipos = pd.DataFrame({
    'Variable': df_subset.columns,
    'Tipo de Dato': df_subset.dtypes.values.astype(str)
})

print("Tipos de Datos en el DataFrame:")

# Mostrar la tabla interactiva de tipos
show(df_tipos)

Tipos de Datos en el DataFrame:


### 3.1. Inspección de Valores Nulos
Antes de eliminar o reemplazar registros, realizaremos un diagnóstico para cuantificar cuántos valores nulos (`NaN`) o celdas vacías existen actualmente en cada variable de nuestro subconjunto.

In [58]:
# Contar la cantidad de valores nulos por columna en el subconjunto
conteo_nulos = df_subset.isnull().sum()
porcentaje_nulos = (conteo_nulos / len(df_subset)) * 100

# Crear un DataFrame resumen para una visualización
df_diagnostico_nulos = pd.DataFrame({
    'Variable': conteo_nulos.index,
    'Valores Nulos': conteo_nulos.values,
    'Porcentaje (%)': porcentaje_nulos.values.round(2)
}).sort_values(by='Valores Nulos', ascending=False)

# Mostrar la tabla interactiva de diagnóstico
show(df_diagnostico_nulos)

### 3.2. Análisis de la Variable Objetivo (`nivel_anemia`)
Verificaremos si la variable dependiente $Y$ (`HC57`) contiene valores nulos o códigos de excepción antes de proceder con el tratamiento de limpieza general.

In [59]:
# Conteo de nulos directos en la variable objetivo
nulos_y = df_subset['nivel_anemia'].isnull().sum()
total_registros = len(df_subset)

# Ver la frecuencia de los valores actuales en la variable objetivo (incluyendo códigos raros si los hubiera)
frecuencia_y = df_subset['nivel_anemia'].value_counts(dropna=False).sort_index()

print("Variable Objetivo (nivel_anemia):")
print(f"   - Total de registros en el dataset: {total_registros}")
print(f"   - Valores nulos (NaN) detectados inicialmente: {nulos_y}")
print("\nDistribución actual de categorías en 'nivel_anemia':")
show(frecuencia_y)

Variable Objetivo (nivel_anemia):
   - Total de registros en el dataset: 19298
   - Valores nulos (NaN) detectados inicialmente: 0

Distribución actual de categorías en 'nivel_anemia':


### 3.3. Tratamiento de Códigos Especiales en la Variable Objetivo
Durante el análisis detectamos que la categoría `9` agrupa a los registros sin medición de hemoglobina o con datos perdidos, lo esperado es el rango `1:4`. Estos registros deben ser excluidos del conjunto de entrenamiento, ya que no aportan una etiqueta válida de clase (`nivel_anemia`).

In [60]:
# 1. Reemplazar el código '9' de la variable objetivo por NaN (para unificarlo como valor perdido)
df_subset['nivel_anemia'] = df_subset['nivel_anemia'].replace(9, np.nan)

# 2. Eliminar estrictamente las filas donde 'nivel_anemia' sea nulo (NaN)
df_clean = df_subset.dropna(subset=['nivel_anemia']).copy()

# 3. Verificar la distribución limpia de la variable objetivo
print("Distribución final de la Variable Objetivo (nivel_anemia) depurada:")
distribucion_limpia = df_clean['nivel_anemia'].value_counts().sort_index()
show(distribucion_limpia)

print(f"\nTotal de registros listos para el modelado tras la limpieza: {len(df_clean)}")

Distribución final de la Variable Objetivo (nivel_anemia) depurada:



Total de registros listos para el modelado tras la limpieza: 17616


### 3.4. Planificación y Validación de Rangos Plausibles para las Variables Predictoras ($X$)
Estableceremos filtros de rangos válidos y códigos de excepción basados en la documentación oficial del INEI. Analizaremos y depuraremos las variables agrupadas en los siguientes bloques:

#### Grupo 1: Variables Antropométricas Básicas
* **`edad_meses` (`HC1`):** 
  * *Rango oficial:* De `0` a `59` meses.
  * *Acción:* Cualquier valor fuera de este intervalo se considerará inconsistente y se eliminará.
* **`peso_kg` (`HC2`):** 
  * *Rango oficial:* De `15` a `500` (con 1 decimal implícito, equivalente a 1.5 kg a 50.0 kg).
  * *Acción:* Filtrar valores fuera del rango biológico y excluir códigos de error.
* **`talla_cm` (`HC3`):** 
  * *Rango oficial:* De `400` a `1500` (con 1 decimal implícito, equivalente a 40.0 cm a 150.0 cm).
  * *Acción:* Mantener únicamente medidas antropométricas válidas para menores de 5 años.

#### Grupo 2: Indicadores Estándar OMS (Puntajes Z)
* **`z_talla_edad` (`HC70`), `z_peso_edad` (`HC71`), `z_peso_talla` (`HC72`), `z_imc` (`HC73`):** 
  * *Rango oficial:* De `-450` a `500`.
  * *Códigos de excepción a limpiar (`NaN`):*
    * `9996` = Altura fuera de los límites plausibles.
    * `9997` = Edad en días fuera de los límites plausibles.
    * `9998` = Casos marcados / valores perdidos.

#### Grupo 3: Variables Demográficas y del Entorno Familiar
* **`sexo_nino` (`HC27`):** 
  * *Valores válidos:* `1` (Hombre) y `2` (Mujer).
* **`educacion_madre_nivel` (`HC61`):** 
  * *Valores válidos:* `0` (Sin educación), `1` (Primaria), `2` (Secundaria), `3` (Superior).
  * *Código de excepción a limpiar:* `8` (No sabe).
* **`educacion_madre_anios` (`HC62`) y `orden_nacimiento` (`HC64`):** 
  * *Acción:* Auditoría de valores atípicos y omisiones según los registros operativos del hogar.

In [61]:
# Grupo 1: Variables Antropométricas Básicas (HC1, HC2, HC3)
variables_grupo1 = ['edad_meses', 'peso_kg', 'talla_cm']

print("[GRUPO 1] Inspección de Estadísticas Descriptivas (Min / Max / Media):")
print("Esto nos permitirá detectar si hay valores fuera de rango o códigos ocultos (ej. 9998).\n")

# Generar tabla resumen con los estadísticos clave
resumen_g1 = df_clean[variables_grupo1].describe().T[['min', 'max', 'count']]
show(resumen_g1)

[GRUPO 1] Inspección de Estadísticas Descriptivas (Min / Max / Media):
Esto nos permitirá detectar si hay valores fuera de rango o códigos ocultos (ej. 9998).



In [62]:
# Definir copias para trabajar de forma segura
df_clean_g1 = df_clean.copy()

# 1. Aplicar filtros de rangos oficiales y excluir códigos de error (como 9999) para el Grupo 1
# - edad_meses: 0 a 59 meses
# - peso_kg: 15 a 500 (1.5 kg a 50.0 kg)
# - talla_cm: 400 a 1500 (40.0 cm a 150.0 cm)

condicion_g1 = (
    (df_clean_g1['edad_meses'] >= 0) & (df_clean_g1['edad_meses'] <= 59) &
    (df_clean_g1['peso_kg'] >= 15) & (df_clean_g1['peso_kg'] <= 500) &
    (df_clean_g1['talla_cm'] >= 400) & (df_clean_g1['talla_cm'] <= 1500)
)

# Filtrar el DataFrame manteniendo solo los registros válidos del Grupo 1
registros_antes_g1 = len(df_clean_g1)
df_clean_g1 = df_clean_g1[condicion_g1].copy()
registros_despues_g1 = len(df_clean_g1)

print("[GRUPO 1] Resultados de Limpieza Antropométrica:")
print(f" - Registros antes de filtrar el Grupo 1: {registros_antes_g1}")
print(f" - Registros válidos tras aplicar rangos oficiales: {registros_despues_g1}")
print(f" - Registros descartados (con códigos 9999 o fuera de rango): {registros_antes_g1 - registros_despues_g1}")

# Actualizar nuestro DataFrame principal de trabajo
df_clean = df_clean_g1

# Verificar las estadísticas descriptivas ya limpiecitas
resumen_g1_limpio = df_clean[variables_grupo1].describe().T[['min', 'max', 'mean', 'count']]
print("\nEstadísticas limpias del Grupo 1:")
show(resumen_g1_limpio)

[GRUPO 1] Resultados de Limpieza Antropométrica:
 - Registros antes de filtrar el Grupo 1: 17616
 - Registros válidos tras aplicar rangos oficiales: 17599
 - Registros descartados (con códigos 9999 o fuera de rango): 17

Estadísticas limpias del Grupo 1:


In [63]:
# Grupo 2: Puntajes Z de la OMS (HC70, HC71, HC72, HC73)
variables_grupo2 = ['z_talla_edad', 'z_peso_edad', 'z_peso_talla', 'z_imc']

print("[GRUPO 2] Inspección de Estadísticas Descriptivas (Min / Max / Media):")
print("Buscando códigos ocultos del INEI (como 9996, 9997, 9998) en los puntajes Z.\n")

# Generar tabla resumen con los estadísticos clave
resumen_g2 = df_clean[variables_grupo2].describe().T[['min', 'max', 'mean', 'count']]
show(resumen_g2)

[GRUPO 2] Inspección de Estadísticas Descriptivas (Min / Max / Media):
Buscando códigos ocultos del INEI (como 9996, 9997, 9998) en los puntajes Z.



In [64]:
# Definir copia para trabajar de forma segura
df_clean_g2 = df_clean.copy()

# 1. Aplicar los rangos oficiales de los Puntajes Z de la OMS (-450 a 500)
# Esto filtra automáticamente los códigos de error/excepción (9996, 9997, 9998)[cite: 1, 2]
condicion_g2 = (
    (df_clean_g2['z_talla_edad'] >= -450) & (df_clean_g2['z_talla_edad'] <= 500) &
    (df_clean_g2['z_peso_edad'] >= -450) & (df_clean_g2['z_peso_edad'] <= 500) &
    (df_clean_g2['z_peso_talla'] >= -450) & (df_clean_g2['z_peso_talla'] <= 500) &
    (df_clean_g2['z_imc'] >= -450) & (df_clean_g2['z_imc'] <= 500)
)

# Filtrar el DataFrame manteniendo solo los registros válidos del Grupo 2
registros_antes_g2 = len(df_clean_g2)
df_clean_g2 = df_clean_g2[condicion_g2].copy()
registros_despues_g2 = len(df_clean_g2)

print("[GRUPO 2] Resultados de Limpieza de Puntajes Z:")
print(f" - Registros antes de filtrar el Grupo 2: {registros_antes_g2}")
print(f" - Registros válidos tras aplicar el rango oficial [-450, 500]: {registros_despues_g2}")
print(f" - Registros descartados (códigos especiales/outliers): {registros_antes_g2 - registros_despues_g2}")

# Actualizar nuestro DataFrame principal de trabajo
df_clean = df_clean_g2

# Verificar las estadísticas descriptivas limpias del Grupo 2
resumen_g2_limpio = df_clean[variables_grupo2].describe().T[['min', 'max', 'mean', 'count']]
print("\nEstadísticas limpias del Grupo 2:")
show(resumen_g2_limpio)

[GRUPO 2] Resultados de Limpieza de Puntajes Z:
 - Registros antes de filtrar el Grupo 2: 17599
 - Registros válidos tras aplicar el rango oficial [-450, 500]: 17527
 - Registros descartados (códigos especiales/outliers): 72

Estadísticas limpias del Grupo 2:


In [65]:
# Grupo 3: Variables Demográficas y del Entorno Familiar
variables_grupo3 = ['sexo_nino', 'educacion_madre_nivel', 'educacion_madre_anios', 'orden_nacimiento']

print("[GRUPO 3] Inspección de Frecuencias de Valores Únicos:")
print("Revisaremos la distribución de cada categoría para detectar códigos de excepción (ej. 'No sabe').\n")

# Mostrar las frecuencias de cada variable del grupo 3 de forma ordenada
for var in variables_grupo3:
    print(f"\n- Distribución de frecuencias para: [{var}]")
    frecuencias = df_clean[var].value_counts(dropna=False).sort_index()
    show(frecuencias)

[GRUPO 3] Inspección de Frecuencias de Valores Únicos:
Revisaremos la distribución de cada categoría para detectar códigos de excepción (ej. 'No sabe').


- Distribución de frecuencias para: [sexo_nino]



- Distribución de frecuencias para: [educacion_madre_nivel]



- Distribución de frecuencias para: [educacion_madre_anios]



- Distribución de frecuencias para: [orden_nacimiento]


In [66]:
# Definir copia para trabajar de forma segura
df_clean_g3 = df_clean.copy()

# 0. Forzar la conversión de las variables del Grupo 3 a tipo numérico
# 'errors="coerce"' convierte cualquier texto o carácter extraño automáticamente en NaN
cols_a_numerico = ['sexo_nino', 'educacion_madre_nivel', 'educacion_madre_anios', 'orden_nacimiento']
for col in cols_a_numerico:
    df_clean_g3[col] = pd.to_numeric(df_clean_g3[col], errors='coerce')

# 1. Aplicar filtros de valores válidos y rangos lógicos para el Grupo 3
condicion_g3 = (
    df_clean_g3['sexo_nino'].isin([1, 2]) &
    df_clean_g3['educacion_madre_nivel'].isin([0, 1, 2, 3]) &
    (df_clean_g3['educacion_madre_anios'] >= 0) & (df_clean_g3['educacion_madre_anios'] <= 25) &
    (df_clean_g3['orden_nacimiento'] >= 1) & (df_clean_g3['orden_nacimiento'] <= 15)
)

# Filtrar el DataFrame manteniendo solo los registros válidos del Grupo 3
registros_antes_g3 = len(df_clean_g3)
df_clean_g3 = df_clean_g3[condicion_g3].copy()
registros_despues_g3 = len(df_clean_g3)

print("[GRUPO 3] Resultados de Limpieza Demográfica y Familiar:")
print(f"  - Registros antes de filtrar el Grupo 3: {registros_antes_g3}")
print(f"  - Registros válidos tras aplicar restricciones: {registros_despues_g3}")
print(f"  - Registros descartados: {registros_antes_g3 - registros_despues_g3}")

# Actualizar nuestro DataFrame principal de trabajo
df_clean = df_clean_g3

# 2. Verificación final de frecuencias limpias para el Grupo 3
print("\n Frecuencias depuradas del Grupo 3:")
for var in variables_grupo3:
    print(f"\n--- Distribución final para: [{var}] ---")
    show(df_clean[var].value_counts().sort_index())

print("Proceso de limpieza de variables predictoras (X) y objetivo (Y) completado")
print(f"Total final de registros limpios: {len(df_clean)}")

[GRUPO 3] Resultados de Limpieza Demográfica y Familiar:
  - Registros antes de filtrar el Grupo 3: 17527
  - Registros válidos tras aplicar restricciones: 16476
  - Registros descartados: 1051

 Frecuencias depuradas del Grupo 3:

--- Distribución final para: [sexo_nino] ---



--- Distribución final para: [educacion_madre_nivel] ---



--- Distribución final para: [educacion_madre_anios] ---



--- Distribución final para: [orden_nacimiento] ---


Proceso de limpieza de variables predictoras (X) y objetivo (Y) completado
Total final de registros limpios: 16476


In [67]:
show(df_clean)

In [68]:
# Crear un resumen limpio de los tipos de datos de cada columna en df_clean
df_tipos = pd.DataFrame({
    'Variable': df_clean.columns,
    'Tipo de Dato': df_clean.dtypes.values.astype(str)
})

print("Tipos de Datos en el DataFrame Limpio:")
print("Todas las variables predictoras (X) y la objetivo (Y) deben ser numéricas (int64 o float64).\n")

# Mostrar la tabla interactiva de tipos
show(df_tipos)

Tipos de Datos en el DataFrame Limpio:
Todas las variables predictoras (X) y la objetivo (Y) deben ser numéricas (int64 o float64).

